# Experiment: Native 32x32 Conv Stem vs 224x224 Upsampling Baseline

- **Purpose:** Compare performance (Loss, Accuracy, Latency, VRAM usage) of native 32x32 CIFAR-10 inputs using modified Conv Stem vs 224x224 upsampling baseline.
- **Reference Plan:** [CIFAR_STEM_EXPERIMENT.md](../agent/experiments/CIFAR_STEM_EXPERIMENT.md)
- **Date:** 2026-08-01


## 1. Setup & Environment


In [6]:
import sys, os, time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.models.build_model import build_resnet18, build_resnet18_cifar_stem, count_trainable_params, count_all_params
from data.dataset import download_cifar10, load_cifar10_dataset
from data.transforms import get_train_transform, get_eval_transform
from data.dataloader import get_cifar10_loaders

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


## 2. Load Datasets & DataLoaders


In [7]:
download_cifar10()
train_loader_224, val_loader_224, test_loader_224 = get_cifar10_loaders(batch_size=64)

train_transform_32 = get_train_transform(resize_size=32, crop_size=32)
eval_transform_32 = get_eval_transform(resize_size=32)

train_set_32 = load_cifar10_dataset(train=True, transform=train_transform_32)
test_set_32 = load_cifar10_dataset(train=False, transform=eval_transform_32)

train_loader_32 = DataLoader(train_set_32, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
test_loader_32 = DataLoader(test_set_32, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
print("DataLoaders initialized successfully!")


DataLoaders initialized successfully!


## 3. Build & Inspect Model Variants


In [8]:
models_dict = {
    "ResNet18_224_Frozen": build_resnet18(num_classes=10, mode="frozen", device=device),
    "ResNet18_224_Finetune": build_resnet18(num_classes=10, mode="finetune", device=device),
    "ResNet18_32_NativeStem_Frozen": build_resnet18_cifar_stem(num_classes=10, mode="frozen", device=device),
    "ResNet18_32_NativeStem_Finetune": build_resnet18_cifar_stem(num_classes=10, mode="finetune", device=device),
}

summary = []
for name, model in models_dict.items():
    summary.append({
        "Model Variant": name,
        "Total Parameters": count_all_params(model),
        "Trainable Parameters": count_trainable_params(model),
        "Input Dim": "224x224" if "224" in name else "32x32"
    })

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))


                  Model Variant  Total Parameters  Trainable Parameters Input Dim
            ResNet18_224_Frozen          11181642                  5130   224x224
          ResNet18_224_Finetune          11181642               8398858   224x224
  ResNet18_32_NativeStem_Frozen          11173962                  6858     32x32
ResNet18_32_NativeStem_Finetune          11173962               8400586     32x32


## 4. Benchmark Function


In [9]:
def train_and_benchmark(model, train_loader, test_loader, epochs=3, lr=1e-3, name=""):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    start_time = time.time()
    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)
    total_time = time.time() - start_time
    model.eval()
    test_loss, test_correct, test_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            test_loss += loss.item() * images.size(0)
            _, preds = outputs.max(1)
            test_correct += preds.eq(labels).sum().item()
            test_total += labels.size(0)
    peak_vram = torch.cuda.max_memory_allocated(device)/(1024**2) if torch.cuda.is_available() else 0.0
    return {
        "Variant": name,
        "Total Time (s)": round(total_time, 2),
        "Time/Epoch (s)": round(total_time/epochs, 2),
        "Peak VRAM (MB)": round(peak_vram, 1),
        "Test Loss": round(test_loss/test_total, 4),
        "Test Acc (%)": round(test_correct/test_total * 100, 2)
    }


## 5. Execute Experiment Benchmarks


In [10]:
results = []
epochs = 3
print("1. ResNet18_224_Frozen...")
results.append(train_and_benchmark(models_dict["ResNet18_224_Frozen"], train_loader_224, test_loader_224, epochs=epochs, lr=1e-3, name="ResNet18_224_Frozen"))
print("2. ResNet18_224_Finetune...")
results.append(train_and_benchmark(models_dict["ResNet18_224_Finetune"], train_loader_224, test_loader_224, epochs=epochs, lr=1e-4, name="ResNet18_224_Finetune"))
print("3. ResNet18_32_NativeStem_Frozen...")
results.append(train_and_benchmark(models_dict["ResNet18_32_NativeStem_Frozen"], train_loader_32, test_loader_32, epochs=epochs, lr=1e-3, name="ResNet18_32_NativeStem_Frozen"))
print("4. ResNet18_32_NativeStem_Finetune...")
results.append(train_and_benchmark(models_dict["ResNet18_32_NativeStem_Finetune"], train_loader_32, test_loader_32, epochs=epochs, lr=1e-4, name="ResNet18_32_NativeStem_Finetune"))

df_res = pd.DataFrame(results)
print("\n=== EXPERIMENT COMPARISON MATRIX ===")
print(df_res.to_string(index=False))


1. ResNet18_224_Frozen...
2. ResNet18_224_Finetune...
3. ResNet18_32_NativeStem_Frozen...
4. ResNet18_32_NativeStem_Finetune...

=== EXPERIMENT COMPARISON MATRIX ===
                        Variant  Total Time (s)  Time/Epoch (s)  Peak VRAM (MB)  Test Loss  Test Acc (%)
            ResNet18_224_Frozen          221.91           73.97           620.2     0.5866         79.76
          ResNet18_224_Finetune          244.22           81.41           716.3     0.2509         91.73
  ResNet18_32_NativeStem_Frozen          128.13           42.71           532.8     1.4648         48.28
ResNet18_32_NativeStem_Finetune          146.80           48.93           610.4     0.7711         72.93
